<a href="https://colab.research.google.com/github/utkuayten/CS401-soffritto/blob/main/optuna_GAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/utkuayten/CS401-soffritto

Cloning into 'CS401-soffritto'...
remote: Enumerating objects: 1968, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 1968 (delta 58), reused 101 (delta 50), pack-reused 1849 (from 1)
Receiving objects: 100% (1968/1968), 797.99 MiB | 26.95 MiB/s, done.
Resolving deltas: 100% (1127/1127), done.
Updating files: 100% (682/682), done.


In [2]:
%pwd

'/content'

In [3]:
%ls

CS401-soffritto/  sample_data/


In [4]:
%cd CS401-soffritto/

/content/CS401-soffritto


In [5]:
%ls

dataset_feature_selection.ipynb  PatchTST/
GAT/                             README.md
GenomicBert/                     run_model.ipynb
iTransformer/                    soffritto/
optuna_BOCO_H1.csv               train_w_parameters.ipynb
optuna_informer_BOC.ipynb        transofritto/
optuna_informer_BOCO.ipynb       transofritto_v2/
optuna_informerV2_BOC.ipynb      wavelet_informer.ipynb
optuna_informerV2_BOCO.ipynb


In [6]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 12.1 MB/s eta 0:00:00


In [7]:
!pip install torch-geometric
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.1.0+cu121.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.5 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.1.0+cu121.html
ERROR: Could not find a version that satisfies the requirement pyg_lib (from versions: none)
ERROR: No matching distribution found for pyg_lib


In [ ]:
!python3 GAT/optuna_tune_gat_intracell.py

[I 2025-12-31 15:47:45,455] A new study created in memory with name: gat_intracell_tune
epoch 001 | train_KL=0.630054 | test_KL=0.556198 | best=0.556198
epoch 010 | train_KL=0.284420 | test_KL=0.244198 | best=0.244198
epoch 020 | train_KL=0.132428 | test_KL=0.117040 | best=0.117040
epoch 030 | train_KL=0.085204 | test_KL=0.082414 | best=0.082414
epoch 040 | train_KL=0.063263 | test_KL=0.066868 | best=0.066868
epoch 050 | train_KL=0.052841 | test_KL=0.056075 | best=0.056075
epoch 060 | train_KL=0.046700 | test_KL=0.049565 | best=0.049565
epoch 070 | train_KL=0.042818 | test_KL=0.046122 | best=0.046122
epoch 080 | train_KL=0.040542 | test_KL=0.044307 | best=0.044307
epoch 090 | train_KL=0.038409 | test_KL=0.044601 | best=0.043586
epoch 100 | train_KL=0.038467 | test_KL=0.042485 | best=0.042370


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

def load_first_npz_array(npz_path: str) -> np.ndarray:
    z = np.load(npz_path)
    return z[z.files[8]]  # exactly like your code

def load_gat_probs_from_npz(npz_path: str) -> np.ndarray:
    z = np.load(npz_path)
    return z["probs"]  # saved as probs in your GAT/predictions/*.npz

def kl_pq_mean_and_per_sample(p_true: torch.Tensor, q_pred: torch.Tensor):
    """
    KL(P||Q) where P=true labels (prob), Q=predictions (prob).
    Shapes: (N, K)
    Returns: (kl_per_sample: (N,), kl_mean: scalar)
    """
    kl_elem = F.kl_div(q_pred.log(), p_true, reduction="none")  # (N, K)
    kl_per_sample = kl_elem.sum(dim=1)                          # (N,)
    kl_mean = kl_per_sample.mean()                              # scalar
    return kl_per_sample, kl_mean


cell_lines = ['H1']
for cell in cell_lines:
    pred = torch.from_numpy(load_gat_probs_from_npz(f"GAT/predictions/{cell}_predictions.npz")).to(torch.float64)
    labels = torch.from_numpy(load_first_npz_array(f"GAT/data/{cell}_labels.npz")).to(torch.float64)

    assert pred.shape == labels.shape, f"Shape mismatch: pred{pred.shape} vs labels{labels.shape}"

    kl_each, kl_avg = kl_pq_mean_and_per_sample(labels, pred)
    print(f"For cell {cell} KL loss: {kl_avg:.5f}")
    print("First 5 KL values:", kl_each[:5], "\n")